# **DPO SU VELVET PER TASK DI NER**

## **1. SETUP DELL'AMBIENTE**

**1.1 INSTALLAZIONE PACCHETTI NECESSARI**

I componenti centrali che useremo sono tre librerie di Hugging Face, ovvero:

- **Transformers** per caricare il modello Velvet

- **TRL** ovvero Transformer Reinforcement Learning, libreria sviluppata da HuggingFace che implementa algoritmi di Reinforcement Learning di modelli linguistici tra cui proprio la Direct Preferences Optimization. TRL funziona da strato che si occupa di tutta la logica del training DPO lasciando allo sviluppatore solo l'onere di fornire i dati e gli iperparametri.

- **PEFT** ovvero Parameter-Efficient Fine-Tuning, altra libreria di HuggingFace che implementa tecniche di fine-tuning efficiente tra cui LoRA (Low Rank Adaptation) che sfrutteremo per




In [ ]:
!pip install -q transformers trl peft bitsandbytes accelerate datasets sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 35.3 MB/s eta 0:00:00


**1.2 IMPORT DEI PACKAGE E CREAZIONE CARTELLA DI LAVORO**

In [ ]:
import os, json, re, shutil, glob
import torch
import pandas as pd
from collections import Counter
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import DPOTrainer, DPOConfig
from datasets import Dataset
from google.colab import drive

os.makedirs('outputs', exist_ok=True) # Creazione cartella 'outputs'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu' # Se disponibile usiamo la GPU, altrimenti la CPU

## **2. PREPARAZIONE DEL DATASET**

 **2.1 CLONAZIONE DEL DATASET KIND DA GITHUB**

 Cloniamo il repository GitHub del dataset KIND e ne visualizziamo la struttura.

 KIND è organizzato in quattro domini:
 - Wikinews
 - Fiction
 - Discorsi di De Gasperi
 - Discorsi di Aldo Moro  

 ciascuno con i propri split train/dev/test in formato TSV (Tab Separated Values)

 Utilizzeremo per tutte le fasi del progetto il dataset Wikinews perché presenta annotazioni gold di alta qualità, un registro giornalistico standard rappresentativo del caso d'uso generale, ed è il dominio più comunemente usato come benchmark nei lavori che utilizzano KIND.

In [ ]:
!git clone https://github.com/dhfbk/KIND.git # Clona il dataset da GitHub
print('\nContenuto della cartella DATASET di KIND:\n')
!find KIND/dataset -type f | sort # Visualizza la struttura della sottocartella 'dataset' di KIND

Cloning into 'KIND'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (101/101), done.
remote: Total 128 (delta 58), reused 68 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 9.34 MiB | 19.45 MiB/s, done.
Resolving deltas: 100% (58/58), done.

Contenuto della cartella DATASET di KIND:

KIND/dataset/degasperi_test.tsv
KIND/dataset/degasperi_train.tsv
KIND/dataset/fiction_test.tsv
KIND/dataset/fiction_train.tsv
KIND/dataset/moro_test.tsv
KIND/dataset/moro_train.tsv
KIND/dataset/wikinews_test.tsv
KIND/dataset/wikinews_train.tsv


**2.2 PARSING DEL DATASET KIND**

Per fare il test baseline usiamo il test set *Wikinews_test.tsv* del dataset KIND.

KIND usa un formato IO puro con i tag = `PER` | `ORG` | `LOC` | `O` (non usa quindi i prefissi 'Begin' [B-] e 'Inside' [I-]).  

Il formato del file è Tab Separated Values: le frasi sono blocchi di righe (separate da una riga vuota) in cui ogni riga è una coppia token/tag (separati da una tabulazione), come nell'estratto qui sotto:

```
Un	O
gruppo	O
di	O
ricercatori	O
dell'	O
università	ORG
di	ORG
Nottingham	ORG
nel	O
Regno	LOC
Unito	LOC
```

Effettuiamo il parsing del dataset per ottenere una lista dove ogni oggetto è un dizionario Python che fa riferimento a una frase, formato come segue:
```
{
    'sentence':      'Un gruppo di ricercatori dell'università di Nottingham nel Regno Unito',  # frase come stringa
    'tokens':        ['Un', 'gruppo', 'di', 'ricercatori', 'dell'', 'università', 'di','Nottingham', 'nel', Regno','Unito'],   # lista di token
    'tags':          ['O', 'O', 'O', 'O', 'O', 'ORG','ORG','ORG',O,'LOC','LOC'],                 # lista di tag IO
    'gold_entities': [{'text': 'università di Nottingham', 'type': 'ORG'},            # entità estratte
                      {'text': 'Regno Unito', 'type': 'LOC'},]                      
    'domain':        'wikinews'                                           # dominio di provenienza
}
```
Il parsing avviene mediante l'utilizzo di tre funzioni in cascata:
```
parse_kind_file()
    └── chiama build_sample() per ogni frase
            └── chiama spans_from_io() per estrarre le entità gold
```
***parse_kind_file*** legge il file riga per riga, accumula token e tag in un buffer, e ogni volta che incontra una riga vuota chiama ***build_sample***

***build_sample*** riceve il buffer di una singola frase e costruisce il dizionario con le chiavi *sentence*, *tokens* e *tags*. Per costruire l'ultima chiave *gold_entities* chiama ***spans_from_io***.

***spans_from_io*** riceve i token e i tag di una frase e li converte in una lista di entità {text, type} ricomponendo i token consecutivi dello stesso tipo in un unico span.

Adottiamo come scelta metodologica quella di filtrare le frasi con meno di 5 token perché non sufficientemente rappresentative (e a rischio di introdurre rumore).

L'output finale è la lista chiamata ***all_samples_filtered***


In [ ]:
# Etichette dei tipi di entità utilizzate in KIND
ENTITY_TYPES = ['PER', 'ORG', 'LOC']

# FUNZIONE PER RICREARE GLI SPAN A PARTIRE DAI TAG IO (INSIDE-OUTSIDE)
# KIND NON USA BIO QUINDI SE UN TOKEN E' INSIDE-OF-ENTITY VIENE RAPPRESENTATO DIRETTAMENTE COL TIPO ("PER", "LOC", "ORG")
# SE INVECE E' OUTSIDE-OF-ENTITY VIENE RAPPRESENTATO CON IL TAG "0"
# OTTENIAMO QUINDI UNA LISTA DELLE ENTITA' NEL FORMATO {text, type}.

def spans_from_io(tokens, tags, entity_types):

    # Inizializza: lista risultati, buffer token correnti, tipo entità corrente
    entities, current_tokens, current_type = [], [], None

    # Iterazione su ogni coppia (parola, etichetta)
    for token, tag in zip(tokens, tags):
        tag = tag.upper().strip() # Tag minuscolo e senza spazi

        # Se il tag è un tipo entità valido (PER, ORG, LOC)
        if tag in entity_types:
            # e se il tipo è lo stesso del tag precedente continua l'entità corrente
            if tag == current_type:
                current_tokens.append(token)
            else:
                # Se il tipo è diverso l'entità precedente (se esiste) è chiusa e viene aggiunta in append a entities
                if current_tokens and current_type:
                    entities.append({'text': ' '.join(current_tokens), 'type': current_type})
                # Inizia una nuova entità col token corrente (il tipo viene aggiornato al tipo corrente)
                current_tokens, current_type = [token], tag
        else:
            # Tag non è un'entità (KIND usa il tag "O"): chiudi l'entità aperta (aggiungendola in append a entities) se presente
            if current_tokens and current_type:
                entities.append({'text': ' '.join(current_tokens), 'type': current_type})
            # Resetta il buffer
            current_tokens, current_type = [], None

    # Fine frase finito il ciclo: salva l'ultima entità rimasta nel buffer (se poresente)
    if current_tokens and current_type:
        entities.append({'text': ' '.join(current_tokens), 'type': current_type})

    return entities  # Restituisce la lista di entità ricostruite

# FUNZIONE CHE COSTRUISCE UN UNICO OGGETTO DIZIONARIO PYTHON CHE CONTIENE
# TUTTI I DATI DI UNA FRASE: TESTO, TOKENS, TAG E ENTITA' ESTRATTE

def build_sample(tokens, tags):
    return {
        'sentence':      ' '.join(tokens),          # Frase come stringa unica
        'tokens':        tokens,                     # Lista dei token originali
        'tags':          tags,                       # Lista dei tag NER corrispondenti
        'gold_entities': spans_from_io(tokens, tags, ENTITY_TYPES),  # Entità gold
    }

# FUNZIONE PER IL PARSING DEL FILE KIND
def parse_kind_file(filepath):
    # Inizializzazione delle liste
    # - sentences: lista con tutte le frasi parsed,
    # - current_token: buffer temporaneo per i token della frase corrente
    # - current_tags: buffer temporaneo per i tag della frase corrente
    sentences, current_tokens, current_tags = [], [], []

    # Apre il file specificando la codifica UTF-8 corretta per il testo in italiano.
    # Il 'with' garantisce che il file venga chiuso automaticamente anche in caso di errore
    with open(filepath, encoding='utf-8') as f:
        for line in f:  # Legge il file una riga alla volta

            # Rimuove il newline finale (\n) che Python include leggendo riga per riga.
            # Senza questo il tag dell'ultima colonna avrebbe \n attaccato
            line = line.rstrip('\n')

            # Riga vuota all'interno di KIND identifica la fine di una frase.
            if line.strip() == '':
                # Salva la frase solo se il buffer non è vuoto
                # (evita di salvare frasi vuote in caso di righe vuote consecutive)
                if current_tokens:
                    sentences.append(build_sample(current_tokens, current_tags))
                    current_tokens, current_tags = [], []  # Resetta i buffer per la prossima frase
                continue  # Salta al prossimo ciclo senza eseguire il codice sotto

            # Divide la riga nelle sue colonne: prima prova con tab,
            # altrimenti usa lo spazio come separatore. KIND usa il formato tsv (Tab Separated Value)
            # quindi il controllo sullo spazio è ridondante ma lo lasciamo per avere una funzione più robusta
            # in caso di utilizzo di un dataset di formato differente.
            parts = line.split('\t') if '\t' in line else line.split()

            # Controlla che la riga abbia almeno 2 colonne (token + tag),
            # per ignorare eventuali righe malformate. Il dataset che usiamo è ben formato, ma si tratta di un controllo in più.
            if len(parts) >= 2:
                current_tokens.append(parts[0])   # Prima colonna → token
                current_tags.append(parts[-1])    # Ultima colonna → tag NER (scegliamo l'ultima con indice "-1" così il codice rimane robusto anche se ci sono colonne intermedie)

        # Gestisce l'ultima frase del file: i file di KIND non sempre terminano
        # con una riga vuota, quindi l'ultima frase potrebbe essere ancora nel buffer
        if current_tokens:
            sentences.append(build_sample(current_tokens, current_tags))

    return sentences  # Lista di tutti i campioni, ognuno costruito da build_sample

# SCEGLIAMO I DATASET DA USARE PER L'INFERENZA
test_files = {
    'wikinews':  'KIND/dataset/wikinews_test.tsv',
    #'fiction':   'KIND/dataset/fiction_test.tsv',
    #'degasperi': 'KIND/dataset/degasperi_test.tsv',
    #'moro':      'KIND/dataset/moro_test.tsv',
}

# Lista che accumula tutti i campioni di tutti i domini
all_samples = []

for domain, filepath in test_files.items():
    # Parsa il file di KIND e ottiene la lista di campioni
    samples = parse_kind_file(filepath)

    # Aggiunge il nome del dominio a ogni campione,
    # utile per analisi successive per dominio
    for s in samples:
        s['domain'] = domain

    # Aggiunge i campioni di questo dominio alla lista generale
    all_samples.extend(samples)
    print('DATASET UTILIZZATO PER L\'INFERENZA')
    print(f'{filepath}: {len(samples)} frasi')

# Filtra le frasi con meno di 5 token, considerate troppo corte
# per essere significative per il task NER
all_samples_filtered = [s for s in all_samples if len(s['tokens']) >= 5]
print(f'\nEsempi nel dataset: {len(all_samples)} \nEsempi con meno di 5 token: {len(all_samples)-len(all_samples_filtered)}\nEsempi totali filtrati: {len(all_samples_filtered)}')

# Conta quante volte appare ogni tipo di entità nel gold standard
# La generator expression itera su tutti i campioni e tutte le loro entità
counter = Counter(e['type'] for s in all_samples_filtered for e in s['gold_entities'])

print('\nDistribuzione entità gold:')
# most_common() restituisce i tipi in ordine decrescente di frequenza
for k, v in counter.most_common():
    print(f'{k}: {v}')

DATASET CHE VERRANNO UTILIZZATI PER L'INFERENZA
KIND/dataset/wikinews_test.tsv: 2594 frasi

Esempi nel dataset: 2594 
Esempi con meno di 5 token: 187
Esempi totali filtrati: 2407

Distribuzione entità gold:
LOC: 1195
ORG: 1055
PER: 1047


# **3. METRICHE BASELINE**

**3.1 CARICAMENTO DEL LLM VELVET 2B**

In [ ]:
# VARIABILE GLOBALE: IDENTIFICATIVO DEL MODELLO VELVET (2 MILIARDI DI PARAMETRI)
MODEL_NAME = 'almawave/velvet-2b'

# CARICAMENTO DEL TOKENIZER: IL TOKENIZER CONVERTE IL TESTO IN TOKEN ID NUMERICI E SEGMENTA IN SUBWORDS LE PAROLE STATISTICAMENTE RARE O LUNGHE
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
# Se il modello non ha un token di padding usiamo il token di End-Of-Sentence per fare il padding
# In realtà Velvet lo prevede ed è il token <pad> con id=3. Lasciamo questa riga come buona pratica in caso il software dovesse essere utilizzato con un modello diverso
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# CARICAMENTO DEL MODELLO: CausalLM perché stiamo caricando un LLM generativo (e non un classificatore)
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16, # usiamo mezza precisione per risparmiare RAM
    device_map='auto', # distribuisce automaticamente il modello su GPU/CPU disponibili
    trust_remote_code=True, # permette di eseguire codice custom del modello da HuggingFace
)
model_base.eval() # Disabilitiamo i componenti usati solo durante il training (come ad esempio il dropout), rendendo le predizioni deterministiche e più veloci
print('\nModello caricato.')


config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.45G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]


Modello caricato.


**3.2 PROMPT CANDIDATI**

Poiché il prompt influenza significativamente le prestazioni di LLM generativo come Velvet su task di NER, conduciamo una fase di prompt selection a partire da un set di cinque prompt candidati con caratterstiche diverse:

1) Prompt sintetico (breve)
2) Prompt mediamente dettagliato (media lunghezza)
3) Prompt molto dettagliato (lungo)
4) Prompt modalità few-shot (con 3 esempi di risposta)


In [ ]:
# Definiamo il dizionario PROMPTS che racchiude funzioni anonime (con argomento 'sentence' ovvero la frase che di volta in volta preleviamo dal dataset parsato)
PROMPTS = {
    # PROMPT SINTETICO
    'P1_sintetico': lambda sentence: tokenizer.apply_chat_template([
        {'role': 'system', 'content': (
            "Estrai persone (PER), organizzazioni (ORG) e luoghi (LOC) dalla frase.\n"
            'Rispondi solo con JSON: {"entities": [{"text": "...", "type": "PER/ORG/LOC"}]}'
        )},
        {'role': 'user', 'content': f'Frase: {sentence}'},
    ], tokenize=False, add_generation_prompt=False),
    # PROMPT MEDIAMENTE DETTAGLIATO
    'P2_medio': lambda sentence: tokenizer.apply_chat_template([
        {'role': 'system', 'content': (
            "Sei un sistema di Named Entity Recognition per l'italiano.\n"
            "Data una frase, estrai tutte le entità delle categorie: "
            "PER (persona), ORG (organizzazione), LOC (luogo/location).\n"
            'Rispondi SOLO con un oggetto JSON valido, senza testo aggiuntivo, nel formato:\n'
            '{"entities": [{"text": "<testo entità>", "type": "<PER|ORG|LOC>"}]}\n'
            'Se non ci sono entità, rispondi: {"entities": []}'
        )},
        {'role': 'user', 'content': f'Frase: {sentence}'},
    ], tokenize=False, add_generation_prompt=False),
    # PROMPT MOLTO DETTAGLIATO
    'P3_verboso': lambda sentence: tokenizer.apply_chat_template([
        {'role': 'system', 'content': (
            "Sei un sistema di Named Entity Recognition per l'italiano.\n"
            "Per ogni frase che ti verrà data:\n"
            "1. Identifica le entità di tipo persona associandole all'etichetta PER, "
            "le entità di tipo organizzazione associandole all'etichetta ORG "
            "e le entità di tipo luogo associandole all'etichetta LOC.\n"
            "2. Restituisci SOLO il JSON finale, senza mostrare il ragionamento "
            "e senza aggiungere alcun testo al di fuori del JSON stesso.\n"
            "Questo è il formato che devi usare:\n"
            '{"entities": [{"text": "<testo>", "type": "<PER|ORG|LOC>"}]}\n'
            'Se non ci sono entità rispondi testualmente così: {"entities": []} '
            'Non inventare nulla e non scrivere nulla che non sia JSON.'
        )},
        {'role': 'user', 'content': f'Frase: {sentence}'},
    ], tokenize=False, add_generation_prompt=False),
    # PROMPT FEW SHOT
    'P4_few_shot': lambda sentence: tokenizer.apply_chat_template([
        {'role': 'system', 'content': (
            "Sei un sistema di Named Entity Recognition per l'italiano.\n"
            "Estrai le entità PER (persona), ORG (organizzazione), LOC (luogo) "
            "e rispondi SOLO con JSON nel formato: "
            '{"entities": [{"text": "...", "type": "..."}]}\n\n'
            f"Esempi:\n{make_few_shot_text()}"
        )},
        {'role': 'user', 'content': f'Frase: {sentence}'},
    ], tokenize=False, add_generation_prompt=False),
}

Definiamo gli strumenti necessari a costruire la parte del Prompt 4 (few-shot) che contiene gli esempi per Velvet, prelevati dal training set di Wikinews (e non dal test set, per non contaminare la valutazione). Gli esempi vengono formattati come coppie *frase, JSON* e inseriti nel system prompt tramite la funzione make_few_shot_text(). Per coprire i casi più rilevanti selezioniamo:

*   Una frase con tutti e tre i tipi di entità (PER, ORG, LOC)
*   Una frase con un solo tipo di entità
*   Una frase senza entità (per mostrare a Velvet come rispondere quando non ce ne sono)


In [ ]:
# Parsing del dataset di TRAINING
train_samples_raw = parse_kind_file('KIND/dataset/wikinews_train.tsv')

# Lista per gli esempi few_shot
few_shot_examples = []

# 1. Una frase con tutti e 3 i tipi
for s in train_samples_raw:
    types_in_s = {e['type'] for e in s['gold_entities']}
    if {'PER', 'ORG', 'LOC'}.issubset(types_in_s):
        few_shot_examples.append(s)
        break

# 2. Una frase con un solo tipo
for s in train_samples_raw:
    types_in_s = {e['type'] for e in s['gold_entities']}
    if len(types_in_s) == 1 and s not in few_shot_examples:
        few_shot_examples.append(s)
        break

# 3. Una frase senza entità
for s in train_samples_raw:
    if not s['gold_entities'] and len(s['tokens']) >= 8 and s not in few_shot_examples:
        few_shot_examples.append(s)
        break

print("Esempi few-shot selezionati:\n")
for ex in few_shot_examples:
    print(f"  Frase: {ex['sentence']}")
    print(f"  Gold:  {ex['gold_entities']}")
    print()

def make_few_shot_text():
    lines = []
    for ex in few_shot_examples:
        answer = json.dumps({'entities': ex['gold_entities']}, ensure_ascii=False)
        lines.append(f"Frase: {ex['sentence']}\n{answer}")
    return "\n\n".join(lines)

Esempi few-shot selezionati:
  Frase: Tra le opere esposte spiccano l' Istituto DESI e l' Istituto METI ( progetti dell' Arch. Anna Heringer ) – realizzati in Bangladesh – per l' utilizzo della terra cruda , paglia , bambù reperiti localmente e un buon sistema di ventilazione della copertura ;
  Gold:  [{'text': 'Istituto DESI', 'type': 'ORG'}, {'text': 'Istituto METI', 'type': 'ORG'}, {'text': 'Anna Heringer', 'type': 'PER'}, {'text': 'Bangladesh', 'type': 'LOC'}]

  Frase: In fiamme l' Istituto Lama Tzong Khapa , monastero buddhista
  Gold:  [{'text': 'Istituto Lama Tzong Khapa', 'type': 'LOC'}]

  Frase: Si sarebbe trattato di un corto circuito nell' impianto elettrico del gompa , la sala di meditazione .
  Gold:  []



**3.3 COSTRUZIONE DELLA FUNZIONE DI INFERENZA**

***parse_json_output*** prende la stringa grezza prodotta da Velvet e la converte in una lista di entità. Lo fa in tre passaggi:

1. Cerca un blocco JSON formato {...} nella stringa con regex e se non lo trova restituisce una lista vuota
2. Converte il JSON trovato in un dizionario Python
3. Scorre la lista entities e scarta elementi malformati o con testo vuoto, raccogliendo solo le entità con tipo valido (PER, ORG, LOC)

Se qualcosa va storto restituisce la lista vuota.

***run_inference_prompt*** esegue l'inferenza in batch su una lista di campioni. Per ogni batch:

- Costruisce i prompt chiamando prompt_fn su ogni frase
- Tokenizza il batch con padding
- Genera l'output con il modello
- Decodifica solo i token generati (esclude i token del prompt con outputs[j][input_len:])
- Parsa l'output grezzo con ***parse_json_output*** e salva tutto nel risultato

In [ ]:
def parse_json_output(raw):
    match = re.search(r'\{.*\}', raw, re.DOTALL)  # Cerca il primo blocco {...} nell'output grezzo (re.DOTALL fa attraversare i newline al carattere '.'
    if not match:  # Se non trova nessun JSON
        return []  # restituisce lista vuota senza crashare
    try:
        data = json.loads(match.group())  # Converte la stringa JSON trovata in dizionario Python
        result = []  # Lista che raccoglie le entità valide
        for e in data.get('entities', []):  # Scorre la lista 'entities' del dizionario. Se la chiave non esiste usa [] come default
            if not isinstance(e, dict) or not e.get('text', '').strip():  # Scarta l'elemento se non è un dizionario oppure se il campo 'text' è assente, vuoto o fatto solo di spazi
                continue
            etype = e.get('type', '').upper().strip()  # Estrae il tipo, lo converte in maiuscolo e rimuove spazi
            if etype in ENTITY_TYPES:  # Accetta solo PER, ORG, LOC, tutto il resto viene scartato
                result.append({'text': e['text'], 'type': etype})  # Aggiunge l'entità valida alla lista risultato
        return result  # Restituisce la lista di entità estratte
    except (json.JSONDecodeError, TypeError):  # Cattura JSON malformato o tipi inattesi
        return []  # In caso di errore restituisce lista vuota


@torch.no_grad()  # Decoratore Python: disabilita il calcolo dei gradienti che non servono in inferenza

def run_inference_prompt(samples, model, tok, prompt_fn,
                         batch_size=32, max_new_tokens=256):
    # samples: lista di campioni su cui fare inferenza
    # model: modello Velvet caricato
    # tok: tokenizer
    # prompt_fn: funzione che costruisce il prompt
    # batch_size: numero di frasi elaborate insieme
    # max_new_tokens: numero massimo di token che il modello può generare per ogni frase
    results = []  # Lista che raccoglie i risultati
    for i in tqdm(range(0, len(samples), batch_size), desc='Inference'):  # Itera a passi di batch_size con barra di progresso
        batch = samples[i:i+batch_size]  # Estrae il batch corrente dalla lista dei campioni
        prompts = [prompt_fn(s['sentence']) for s in batch]  # Costruisce il prompt per ogni frase del batch chiamando prompt_fn
        inputs = tok(prompts, return_tensors='pt', padding=True,  # Tokenizza tutti i prompt del batch
                     truncation=True, max_length=900).to(DEVICE)  # padding=True allinea le sequenze alla stessa lunghezza, truncation taglia quelle troppo lunghe, max_length=900 è il limite massimo di token in input
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,  # Genera al massimo max_new_tokens token nuovi
            do_sample=False,  # Generazione deterministica: lo stesso input produce sempre lo stesso output
            pad_token_id=tok.eos_token_id,  # Usa il token di fine sequenza come token di padding
        )
        for j, sample in enumerate(batch):  # Itera sui campioni del batch con indice j
            input_len = inputs['input_ids'][j].shape[0]  # Lunghezza del prompt in token per il campione j-esimo
            raw = tok.decode(outputs[j][input_len:], skip_special_tokens=True).strip()  # Decodifica solo i token generati (esclude il prompt) e rimuove spazi
            results.append({**sample, 'raw_output': raw,  # Aggiunge al risultato tutti i campi del campione originale
                            'pred_entities': parse_json_output(raw)})  # più l'output grezzo e le entità parsate
    return results  # Restituisce la lista completa dei risultati

**3.4 COSTRUZIONE DELLA FUNZIONE DI VALUTAZIONE CON PARTIAL MATCH**

Dal momento che Velvet è un LLM generativo e non un classificatore sequenziale, consideriamo valido anche il partial match. La funzione ***compute_metrics_partial*** conteggia quindi come TP anche i casi in cui il testo predetto e il testo gold si contengono a vicenda. Ad esempio "Università di Nottingham" e "università di Nottingham" o "Germania 2006" e "Germania" vengono considerati un match corretto.

Questo approccio tiene conto del fatto che i modelli generativi tendono a riprodurre le entità con piccole variazioni rispetto al gold standard senza che ciò costituisca un errore semantico.

Calcoliamo le metriche sulle singole classi (PER, LOC e ORG) ma anche sulle classi aggregate con logica MICRO, che prevede di aggregare tutti i TP, FP e FN di tutte le classi in un unico conteggio e poi di calcolare le metriche su quei totali. La metrica MICRO è utile come numero unico che riassume le performance globali del modello. Le classi risultano ragionevolmente bilanciate con una preponderanza di LOC:

LOC: 1195 (~36%)

ORG: 1055 (~32%)

PER: 1047 (~32%)

La funzione ***print_metrics*** stampa una tabella con i risultati in questo formato
```
==================================================
Tipo      Precision     Recall         F1
--------------------------------------------------
PER           -            -           -
ORG           -            -           -
LOC           -            -           -
MICRO         -            -           -
==================================================
```

In [ ]:
# Metriche con partial match
def normalize(text):
    return text.lower().strip() # Testo tutto lower case

def is_match(a, b):
    return a == b or a in b or b in a # Regola per matching parziale: c'è match anche se a contiene b o b contiene a

def compute_metrics_partial(results):
    stats = {et: {'tp':0,'fp':0,'fn':0} for et in ENTITY_TYPES} # Variabile globale ENTITY_TYPES = ['PER', 'ORG', 'LOC'] crea una chiave con conteggio tp, fp e fn per ognuna di esse
    stats['MICRO'] = {'tp':0,'fp':0,'fn':0} # Aggiunta della chiave micro per le micro-average
    for r in results:
        gold = [(normalize(e['text']), e['type'].upper()) for e in r['gold_entities']] # Etichette gold trasformate in tuple (testo_normalizzato, tipo) e inserite nella lista gold[]
        pred = [(normalize(e['text']), e['type'].upper()) for e in r['pred_entities']] # Stessa cosa per le predizioni di Velvet inserite nella lista pred[]
        for et in ENTITY_TYPES:
            ge = [g for g in gold if g[1]==et] # ge raccoglie le etichette gold del tipo corrente
            pe = [p for p in pred if p[1]==et] # per etichette le etichette predette del tipo corrente
            mg, mp = set(), set() # usiami per il conteggio dei tag che hanno già matchato dei set Python: non ammettono duplicati e non garantiscono l'ordine. La ricerca è veloce.
            for pi,p in enumerate(pe): # scorre con indice la lista delle predizioni del tipo corrente con pi indice p valore dell'elemento
                for gi,g in enumerate(ge): # scorre con indice la lista delle etochette gold del tipo corrente con gi indice g valore dell'elemento
                    if gi not in mg and is_match(p[0],g[0]): # match se is_match resituisce true AND se l'indice dell'etichetta non è nel set che contiene gli indici delle etichette già matchate
                        mg.add(gi); mp.add(pi); break # salviamo gli indice delle etichette (predetta e golg) che hanno fatto match per non duplicare il match stesso
            stats[et]['tp'] += len(mg) # conteggio dei True Positive
            stats[et]['fp'] += len(pe)-len(mp) # conteggio dei False Positive
            stats[et]['fn'] += len(ge)-len(mg) # conteggio dei False Negative

        # CALCOLO DELLE MICRO STATS
        mg, mp = set(), set() #ri-inizializziamo i due set per il controllo delle etichette già matchate
        for pi,p in enumerate(pred): # scorre con indice la lista intera delle predizioni
            for gi,g in enumerate(gold): # scorre con indice la lista intera delle etichette gold
                if gi not in mg and p[1]==g[1] and is_match(p[0],g[0]): # controllo match con controllo esplicito del tipo p[1] e g[1] sono i tipi di etichetta
                    mg.add(gi); mp.add(pi); break # aggiunta degli indici delle etichette matchate ai set Python
        stats['MICRO']['tp'] += len(mg) # conteggio dei True Positive
        stats['MICRO']['fp'] += len(pred)-len(mp) # conteggio dei False Positive
        stats['MICRO']['fn'] += len(gold)-len(mg) # conteggio dei False Negative
    out = {}
    for key,s in stats.items(): # key è la chiave s il dizionario (spachettamento diretto in due variabili)
        p  = s['tp']/(s['tp']+s['fp']) if (s['tp']+s['fp'])>0 else 0.0 # calcolo precision con controllo eventuale divisione per zero
        r  = s['tp']/(s['tp']+s['fn']) if (s['tp']+s['fn'])>0 else 0.0 # calcolo recall con controllo eventuale divisione per zero
        f1 = 2*p*r/(p+r) if (p+r)>0 else 0.0 #c alcolo F1 score con controllo eventuale divisione per zero
        out[key] = {'precision':round(p,4),'recall':round(r,4),'f1':round(f1,4),**s} # raccolta dei risultati nella lista out con valori approssimati a 4 decimali
    return out

# STAMPA DELA TABELLA DELLE METRICHE
def print_metrics(metrics, label=''):
    print('\n' + label)
    print('\n' + '='*50)
    print(f'{"Tipo":<8} {"Precision":>10} {"Recall":>10} {"F1":>10}')
    print('-'*50)
    for key in [*ENTITY_TYPES,'MICRO']:
        m = metrics[key]
        print(f'{key:<8} {m["precision"]:>10.4f} {m["recall"]:>10.4f} {m["f1"]:>10.4f}') # Stringa formattata con spazi
    print('='*50)

**3.4 INFERENZA CON I 4 PROMPT E CALCOLO DELLE METRICHE**

Richiamiamo la funzione ***run_inference_prompt*** all'interno di un ciclo per eseguirla sui 4 prompt candidati. I risultati avranno la stessa struttura dei campioni parsati dal dataset, con l'aggiunta di due chiavi: *raw_output*, che contiene la risposta testuale grezza di Velvet, e *pred_entities*, che contiene le entità estratte dal parser ***parse_json_output*** in formato lista di dizionari."
```
{
    'sentence':      'Un gruppo di ricercatori dell'università di Nottingham nel Regno Unito',  # frase come stringa
    'tokens':        ['Un', 'gruppo', 'di', 'ricercatori', 'dell'', 'università', 'di','Nottingham', 'nel', Regno','Unito'],  # lista di token
    'tags':          ['O', 'O', 'O', 'O', 'O', 'ORG','ORG','ORG',O,'LOC','LOC'],  # lista di tag IO
    'gold_entities': [{'text': 'università di Nottingham', 'type': 'ORG'},  # entità estratte
                      {'text': 'Regno Unito', 'type': 'LOC'},]                      
    'domain':        'wikinews'  # dominio di provenienza
    "raw_output": "{\"entities\": [{\"text\": \"Università di Nottingham\", \"type\": \"ORG\"}]}", # output di
    "pred_entities": [
      {
        "text": "Università di Nottingham",
        "type": "ORG"
      }
   ]
}
```

Otteniamo in output 4 file con i risultati di ciascun prompt:


*   results_P1_sintetico.json
*   results_P2_medio.json
*   results_P3_verboso.json
*   results_P4_few_shot.json

Sulla base dei risultati calcoliamo le metriche con ***compute_metrics_partial*** e costruiamo un DataFrame per il confronto delle performance: prima raccogliamo i risultati dei 4 prompt in un'unica tabella, poi la pivotattiamo sul campo F1 per visualizzare lo score di ogni prompt per ogni classe e sulle classi aggregate con logica MICRO. Applichiamo infine una heatmap per evidenziare i valori migliori e peggiori e identificare immediatamente il prompt con le prestazioni più alte (da rosso intenso per il valore più basso a verde pieno per il valore più alto).

In [ ]:
# Dizionario che raccoglie le metriche di tutti i prompt
all_prompt_metrics = {}

for prompt_name, prompt_fn in PROMPTS.items():  # Itera sul dizionario PROMPTS: prompt_name è la chiave, prompt_fn è la lambda
    print(f'\n' + '-'*50)
    print(f'Esecuzione: {prompt_name}')  # Stampa il nome del prompt corrente

    parse_fn = parse_json_output  # Funzione di parsing

    # FUNZIONE CHE FA L'INFERENZA
    results = run_inference_prompt(
        all_samples_filtered, model_base, tokenizer,  # Dataset filtrato, modello base, tokenizer
        prompt_fn=prompt_fn, batch_size=32  # Prompt corrente, parser JSON, 32 frasi per batch
    )

    with open(f'outputs/results_{prompt_name}.json', 'w', encoding='utf-8') as f:  # Apre o crea il file di output per il prompt corrente
        slim = [{k: v for k, v in r.items() if k not in ('tokens', 'tags')} for r in results]  # Rimuove i campi 'tokens' e 'tags' per alleggerire il file
        json.dump(slim, f, ensure_ascii=False, indent=2)  # Scrive il JSON su file con indentazione leggibile
    print(f'Risultati salvati nel file results_{prompt_name}.json')

    metrics = compute_metrics_partial(results)  # Calcola Precision, Recall e F1 con partial match
    all_prompt_metrics[prompt_name] = metrics  # Salva le metriche nel dizionario usando il nome del prompt come chiave
    print_metrics(metrics, prompt_name)  # Stampa la tabella delle metriche per il prompt corrente

# Costruisce la lista di righe per il DataFrame comparativo
rows = []
for pname, metrics in all_prompt_metrics.items():  # Itera su tutti i prompt e le loro metriche
    for key in [*ENTITY_TYPES, 'MICRO']:  # Itera su PER, ORG, LOC, MICRO
        m = metrics[key]  # Estrae le metriche per il tipo corrente
        rows.append({
            'Prompt': pname,       # Nome del prompt
            'Tipo':   key,         # Tipo di entità o MICRO
            'P':      m['precision'],  # Precision
            'R':      m['recall'],     # Recall
            'F1':     m['f1'],         # F1 score
        })

df_prompts = pd.DataFrame(rows)  # Crea il DataFrame dalla lista di righe

# Pivot: righe = prompt, colonne = tipo di entità, valori = F1
df_pivot = df_prompts.pivot(index='Prompt', columns='Tipo', values='F1')
df_pivot = df_pivot[[*ENTITY_TYPES, 'MICRO']].sort_values('MICRO')  # Ordina le colonne e le righe per MICRO F1 crescente

print('\n=== F1 per prompt (ordinati per MICRO F1) ===')
display(df_pivot.style
        .background_gradient(cmap='RdYlGn', subset=[*ENTITY_TYPES, 'MICRO'])  # Heatmap verde/rosso sulle colonne F1
        .format('{:.4f}'))  # Formatta i valori con 4 decimali

df_prompts.to_csv('outputs/prompt_comparison.csv', index=False)  # Salva il DataFrame in formato CSV senza l'indice numerico
# shutil.copy('outputs/prompt_comparison.csv', f'{DRIVE_OUTPUTS}/prompt_comparison.csv')  # Copia il CSV su Drive

with open('outputs/all_prompt_metrics.json', 'w') as f:
    json.dump(all_prompt_metrics, f, indent=2)  # Salva tutte le metriche in formato JSON
#   shutil.copy('outputs/all_prompt_metrics.json', f'{DRIVE_OUTPUTS}/all_prompt_metrics.json')  # Copia il JSON su drive


--------------------------------------------------
Esecuzione: P1_sintetico


Inference:   0%|          | 0/76 [00:00<?, ?it/s]

Risultati salvati nel file results_P1_sintetico.json
P1_sintetico


Tipo      Precision     Recall         F1
--------------------------------------------------
PER          0.2587     0.3209     0.2864
ORG          0.1847     0.1374     0.1576
LOC          0.2311     0.0845     0.1238
MICRO        0.2309     0.1765     0.2001

--------------------------------------------------
Esecuzione: P2_medio


Inference:   0%|          | 0/76 [00:00<?, ?it/s]

Risultati salvati nel file results_P2_medio.json
P2_medio


Tipo      Precision     Recall         F1
--------------------------------------------------
PER          0.4862     0.1681     0.2498
ORG          0.2664     0.0616     0.1001
LOC          0.5287     0.0385     0.0718
MICRO        0.4141     0.0870     0.1439

--------------------------------------------------
Esecuzione: P3_verboso


...

Risultati salvati nel file results_P4_few_shot.json
P4_few_shot


Tipo      Precision     Recall         F1
--------------------------------------------------
PER          0.3842     0.4183     0.4005
ORG          0.1387     0.3412     0.1973
LOC          0.2704     0.1967     0.2277
MICRO        0.2244     0.3133     0.2615

--------------------------------------------------
Esecuzione: P5_lista


Inference:   0%|          | 0/76 [00:00<?, ?it/s]

Risultati salvati nel file results_P5_lista.json
P5_lista


Tipo      Precision     Recall         F1
--------------------------------------------------
PER          0.0000     0.0000     0.0000
ORG          0.0000     0.0000     0.0000
LOC          0.0000     0.0000     0.0000
MICRO        0.0000     0.0000     0.0000

=== F1 per prompt (ordinati per MICRO F1) ===


Tipo,PER,ORG,LOC,MICRO
Prompt,,,,
P5_lista,0.0000,0.0000,0.0000,0.0000
P3_verboso,0.0986,0.0588,0.0230,0.0599
P2_medio,0.2498,0.1001,0.0718,0.1439
P1_sintetico,0.2864,0.1576,0.1238,0.2001
P4_few_shot,0.4005,0.1973,0.2277,0.2615


Salvato: outputs/prompt_comparison.csv


# **4. DPO**

**4.1 INFERENZA SUL TRAINING SET**

Essendo risultato P4 (few-shot) il prompt con la F1 score più alta su tutte le etichette e in logica MICRO, eseguiamo l'inferenza sul *wikinews_train.tsv* per estrarre le coppie necessarie al fine-tuning DPO. Utilizziamo il training set invece del test set per evitare il train/test overlap: se costruissimo le coppie DPO dai risultati sul test set, il modello verrebbe addestrato su esempi provenienti dallo stesso insieme su cui viene poi valutato, rendendo le metriche finali non affidabili.

Parsiamo il dataset con ***parse_kind_file***, eseguiamo l'inferenza con ***run_inference_prompt*** usando P4 come prompt e salviamo i risultati nel file *results_train_P4_few_shot.json*."

In [ ]:
# Carica e parsa il training set WN
train_samples_raw = parse_kind_file('KIND/dataset/wikinews_train.tsv')
train_samples_filtered = [s for s in train_samples_raw if len(s['tokens']) >= 5]
print(f'Training set: {len(train_samples_raw)} frasi → filtrate: {len(train_samples_filtered)}')

# Inferenza con P4_few_shot sul training set
results_train = run_inference_prompt(
    train_samples_filtered, model_base, tokenizer,
    prompt_fn=PROMPTS['P4_few_shot'],
    batch_size=64
)

# Salva risultato JSON
with open('outputs/results_train_P4_few_shot.json', 'w', encoding='utf-8') as f:
    slim = [{k: v for k, v in r.items() if k not in ('tokens', 'tags')} for r in results_train]
    json.dump(slim, f, ensure_ascii=False, indent=2)
print(f'Salvati {len(results_train)} risultati in outputs/results_train_P4_few_shot.json')

NameError: name 'parse_kind_file' is not defined

**4.1 COSTRUZIONE COPPIE DPO**

La costruzione delle coppie di esempi per la Direct Preference Optimization avviene secondo questa ide: per ogni frase del training set dove Velvet ha sbagliato, ripetiamo al modello la sua risposta risposta etichettandola come sbagliata e gli affianchiamo quella che invece sarebbe stata la risposta giusta.

Ogni coppia ha quindi campi:

*   *prompt*, cioè la frase formulata con P4_few_shot esattamente come era statapassata al modello durante l'inferenza

*   *chosen*, ovvero il JSON con le entità gold che rappresenta la risposta corretta

*   *rejected*, l'output grezzo che Velvet aveva effettivamente prodotto.

```
{
    "prompt": "<s><instruction>Sei un sistema di Named Entity Recognition per l'italiano.\nEstrai le entità PER (persona), ORG (organizzazione), LOC (luogo) e rispondi SOLO con JSON nel formato: {\"entities\": [{\"text\": \"...\", \"type\": \"...\"}]}\n\nEsempi:\nFrase: Tra le opere esposte spiccano l' Istituto DESI e l' Istituto METI ( progetti dell' Arch. Anna Heringer ) – realizzati in Bangladesh – per l' utilizzo della terra cruda , paglia , bambù reperiti localmente e un buon sistema di ventilazione della copertura ;\n{\"entities\": [{\"text\": \"Istituto DESI\", \"type\": \"ORG\"}, {\"text\": \"Istituto METI\", \"type\": \"ORG\"}, {\"text\": \"Anna Heringer\", \"type\": \"PER\"}, {\"text\": \"Bangladesh\", \"type\": \"LOC\"}]}\n\nFrase: In fiamme l' Istituto Lama Tzong Khapa , monastero buddhista\n{\"entities\": [{\"text\": \"Istituto Lama Tzong Khapa\", \"type\": \"LOC\"}]}\n\nFrase: Si sarebbe trattato di un corto circuito nell' impianto elettrico del gompa , la sala di meditazione .\n{\"entities\": []}\n\nFrase: Un gruppo di ricercatori dell' università di Nottingham nel Regno Unito , guidato da Simon Lee , ha scoperto che i cervelli di alcune specie di locusta e blattoidei contengono sostanze tossiche per gli organismi resistenti agli antibiotici .</instruction>",
    "chosen": "{\"entities\": [{\"text\": \"università di Nottingham\", \"type\": \"ORG\"}, {\"text\": \"Regno Unito\", \"type\": \"LOC\"}, {\"text\": \"Simon Lee\", \"type\": \"PER\"}]}",
    "rejected": "{\"entities\": [{\"text\": \"Università di Nottingham\", \"type\": \"ORG\"}, {\"text\": \"ricercatori\", \"type\": \"PER\"}, {\"text\": \"Simon Lee\", \"type\": \"PER\"}, {\"text\": \"specie di locusta\", \"type\": \"PER\"}, {\"text\": \"blattoidei\", \"type\": \"ORG\"}]}"
  },
  ```
Le frasi per le quali Velvet aveva già dato una risposta corretta vengono escluse perché non portano informazione utile al training: il modello sa già gestire bene quei casi.

È importante che il prompt nelle coppie DPO usi lo stesso formato usato in inferenza (quindi sempre il *P4_few_shot*) altrimenti il modello viene addestrato su un formato e valutato su un altro, rendendo il fine-tuning inefficace.

In [ ]:
# Carica i risultati dell'inferenza sul training set salvati in precedenza
with open('outputs/results_train_P4_few_shot.json', encoding='utf-8') as f:
    results_P4_few_shot = json.load(f)

# Ricalcola pred_entities dal raw_output — il file salvato potrebbe non averle
for r in results_P4_few_shot:
    r['pred_entities'] = parse_json_output(r['raw_output'])  # Parsa l'output grezzo e aggiunge le entità predette

print(f'Caricati {len(results_P4_few_shot)} risultati')

def build_dpo_pairs(results):
    pairs = []  # Lista che accumula le coppie DPO
    for r in results:  # Itera su ogni campione
        gold_set = {(normalize(e['text']), e['type'].upper()) for e in r['gold_entities']}  # Insieme delle entità gold come tuple (testo_normalizzato, tipo)
        pred_set = {(normalize(e['text']), e['type'].upper()) for e in r['pred_entities']}  # Stesso formato per le entità predette
        if gold_set == pred_set:  # Se la predizione è già corretta
            continue  # Salta — non serve correggere nulla
        chosen_json  = json.dumps({'entities': r['gold_entities']}, ensure_ascii=False)  # La risposta corretta è il JSON con le entità gold
        rejected_out = r['raw_output'].strip() if r['raw_output'].strip() else '{"entities": []}'  # La risposta errata è l'output grezzo di Velvet. Se è vuoto usa JSON vuoto come fallback
        pairs.append({'prompt': PROMPTS['P4_few_shot'](r['sentence']),  # Costruisce il prompt con P4_few_shot — deve essere lo stesso usato in inferenza
              'chosen': chosen_json, 'rejected': rejected_out})  # Aggiunge la coppia (prompt, chosen, rejected) alla lista
    return pairs  # Restituisce la lista di coppie DPO

dpo_pairs = build_dpo_pairs(results_P4_few_shot)  # Costruisce le coppie dal training set
print(f'Coppie DPO: {len(dpo_pairs)} / {len(results_P4_few_shot)}')  # Mostra quante coppie sono state generate sul totale dei campioni

with open('outputs/dpo_pairs.json', 'w', encoding='utf-8') as f:
    json.dump(dpo_pairs, f, ensure_ascii=False, indent=2)  # Salva le coppie su file JSON con indentazione leggibile

Caricati 9980 risultati
Coppie DPO: 6883 / 9980


**4.2 CONFIGURAZIONE BITSANDBYTES E LoRA**

Per il fine tuning  DPO è necessario impostare due configurazioni importanti:

1.   *CONFIGURAZIONE BITSANDBYTES*

Utilizziamo una configurazione BitsAndBites a 4-bt invece che a 16-bit in modo da ridurre il consumo di memoria.
Per i pesi usiamo la quantizzazione *nf4* (usa 4 bit e può quindi rappresentare 16 valori), che ci permettere di comprimere rimanendo sufficientemente accurata. Per i calcoli durante il forward pass usiamo *bfloat_16* che perde un po' di precisione sulla mantissa (7 bit anziché i 10 bit di *float16*) ma rimane robusto su esponenti molto grandi o molto piccoli (8 bit dedicati all'espondente rispetto ai 5 di *float16*) di fatto rendendo quasi impossibile l'overflow.

*double_quant* quantizza anche i parametri di quantizzazione stessi, risparmiando ulteriore memoria.

2.   *CONFIGURAZIONE LoRA*

La tecnica *LoRA (Low-Rank Adaptation)* serve per fare fine-tuning di modelli grandi aggiornando solo una piccola frazione dei parametri. Invece di aggiornare tutti i 2 miliardi di parametri di Velvet durante il training (che sarebbe complicato sulla GPU singola a nostra disposizione in ambiente Colab) con LoRA si aggiungono due matrici aggiuntive (A e B) di piccole dimensioni (ovvero con rango basso, da qui il nome) solo per alcuni layer. Questo significa lavorare solo su una piccola frazione minuscola dei parametri totali durante il training.

La classe **LoraConfig** della libreria PEFT definisce la configurazione delle matrici LoRA. Abbiamo utilizzato i seguenti paramerti:

- *r=16*:  rappresenta il rango delle matrici LoRA. Controlla quanti parametri vengono aggiunti e quindi la capacità espressiva dell'adattamento. I valori più comuni sono 8, 16, 32, 64. Abbiamo scelto 16 come valore intermedio, quindi abbastanza espressivo per un task NER ma non così alto da rischiare overfitting su un dataset relativamente piccolo come le coppie DPO.

- *lora_alpha=32*: fattore di scala dell'aggiornamento LoRA (regola il peso dell'adattamento LoRA sul modello base). La convenzione più diffusa è quella di impostarlo al doppio di r, per questo è impostato a 32.

- *target_modules*: i layer su cui applicare LoRA. Velvet è basato su architettura Mistral, una famiglia di LLM sviluppata dalla startup francese Mistral AI. La configutrazione LoRa pù diffusa per questo tipo di architettura  prevede di includere tutti i layer di attenzione ovvero
*q_proj* (query), *k_proj* (key), *v_proj* (value), *o_proj* (output) e tutti i layer feed-forward ovvero *gate_proj*, *up_proj*, *down_proj*.

- *lora_dropout=0.05*: il dropout prevede di "spegnere" casualmente alcuni neuroni per evitare che il modello si adatti troppo ai dati (overfitting). In questo caso è applicato alle matrici LoRA durante il training. Dunque i pesi delle matrici LoRA hanno un 5% di probabilità di essere azzerati. Si tratta di un valore conservativo, abbastanza per aggiungere regolarizzazione senza penalizzare l'apprendimento.

- *bias='none'* prevede di non aggiornare i bias del modello, ma solo i pesi. È la scelta standard con LoRA perché i bias hanno pochi parametri e aggiornarli porta benefici trascurabili.

- *task_type=TaskType.CAUSAL_LM* specifica che stiamo facendo l'adattamento su un Language Model causale (ovvero Velvet stesso). Serve alla libreria PEFT per configurare correttamente la struttura LoRA.

La funzione **get_peft_model** carica il modello base e gli applica la configurazione, congelando i pesi originali e aggiungendo le matrici Low Rank.

**print_trainable_parameters()** mostra quanti parametri verranno effettivamente aggiornati durante il training, tipicamente meno dell'1% del totale.

Prima dell'addestramento cancelliamo il modello base per liberare memoria.

In [ ]:
del model_base
torch.cuda.empty_cache()
print('Memoria liberata.')


Memoria liberata.


In [ ]:
#CONFIGURAZIONE BITSANDBYTES
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Caricamento a 4 bit
    bnb_4bit_quant_type='nf4', # Quantizzazione dei pesi (4 bit)
    bnb_4bit_compute_dtype=torch.bfloat16, # Formato numerico per i calcoli: 7 bit per la mantissa, 8 bit per l'esponente (1 bit per il segno)
    bnb_4bit_use_double_quant=True, # Doppia quantizzazione
)

#CONFIGURAZIONE LORA
lora_config = LoraConfig(
    r=16, lora_alpha=32, # Rango e fattore di scala
    target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'], # Lauer su cui applicare le matrici
    lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM, # Dropout (5%), bias (esclusi dall'adattamento) e tipo di task (fine-tuning LLM causale)
)

# CARICAMENTO  DEL MODELLO
model_for_dpo = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, # Configurazione BNB definita sopra
    device_map='auto', trust_remote_code=True,
)
model_for_dpo = get_peft_model(model_for_dpo, lora_config) # Carica il modello e applica la configurazione
model_for_dpo.print_trainable_parameters() # Stampa del numero parametri su cui verrà fatto il traning


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

trainable params: 19,726,336 || all params: 2,242,824,192 || trainable%: 0.8795


**4.3 FINE TUNING DPO**

Si compone di 3 fasi

**1. Preparazione dei dati**

Le coppie DPO costruite nella cella precedente vengono convertite in un oggetto Dataset di HuggingFace perché questo è il formato che la classe **DPOTrainer** si aspetta. Il dataset viene poi diviso in training set (90%) ed evaluation set (10%). Questo ci serve a monitorare eventuale overfitting: se la loss sull'evaluation set smette di scendere e riprende a salire il modello va in overfitting.
Una curiosità su seed=42: il dataset viene diviso con una procedura nominalmente randomica ma sappiamo che in informatica questo non è possibile. La generazione è sempre pseudo-casuali, calcolata a partire un intero detto *seed*. Il valore 42 è una scelta classica nel mondo del ML e viene dal romano *Guida galattica per autostoppisti* dove il numero 42 è la risposta a tutto. L'intero scelto in sé e per sé non cambia nulla, l'importante è che venga scelto lo stesso intero se si vuole ricondurre l'esperimento con la stessa suddivisione del dataset.

**2. Configurazione del training**

La classe **DPOConfig** raccoglie tutti gli iperparametri, commentati nel codice. I 3 più importanti sono

- *beta*: parametro che controlla quanto il modello può allontanarsi da quello base durante l'aggiornamento.

- *learning_rate=*: dimensione del passo con cui i pesi del modello vengono aggiornati ad ogni iterazione (più è alto, più l'aggiornamento è aggressivo).

- *num_train_epochs* numero di epoche, ovvero numero di volte che il modello vede l'intero dataset di training durante l'addestramento.

In un primo tentativo avevamo previsto:
```
-num_train_epochs=3
-learning_rate=5e-5
-beta=0.1
```
ma il risultato è stato un overfitting evidente: la F1 migliorava ma la Precision crollava, e soprattutto il modello allucinava su frasi senza entità  producendo entità inesistenti: aveva appreso erroneamente la regola "produci sempre qualcosa".

I risultati attuali sono con:

```
-num_train_epochs=1
-learning_rate=1e-5
-beta=0.5
```
ovvero una sola epoca per ridurre l'esposizione ai dati di training e limitare l'overfitting, un learning rate più basso per un aggiornamento più conservativo e un parametro beta che tiene il modello più vicino a quello base.

Fra gli altri iperparametri, mantenuti inalterati abbiamo:

- *gradient_accumulation_steps=2* con *batch_size=4* simula un batch effettivo di 8 coppie per learning step (2x4).

- *warmup_steps=50* sono gli step di "riscaldamento, che fanno crescere il learning rate gradualmente nei primi 50 step invece di partire subito al massimo, stabilizzando le fasi iniziali del training.

- *load_best_model_at_end=True* garantisce che alla fine venga caricato il checkpoint con la loss di valutazione più bassa, non necessariamente l'ultimo.

**3. Esecuzione del Training**

La classe **DPOTrainer** riceve il modello con LoRA applicato, la configurazione e i dataset. Utilizziamo il parametro *ref_model=None* perché con QLoRA il modello base congelato funge già da riferimento implicito e non serve caricare una copia separata.

Dopo il training vengono salvati sia i pesi dell'adapter LoRA che il tokenizer nella stessa cartella, così il modello può essere ricaricato completamente in una sessione successiva.

In [ ]:
*# Blocco commentato: ricarica le coppie DPO da Drive se la variabile dpo_pairs non è in memoria
# Utile in caso di reset della sessione Colab
'''
if 'dpo_pairs' not in dir():
    with open(f'{DRIVE_OUTPUTS}/dpo_pairs.json', encoding='utf-8') as f:
        dpo_pairs = json.load(f)
    print(f'Ricaricate {len(dpo_pairs)} coppie da file.')
'''

dpo_dataset = Dataset.from_list(dpo_pairs)  # Converte la lista di coppie DPO in un oggetto Dataset di HuggingFace
dpo_split = dpo_dataset.train_test_split(test_size=0.1, seed=42)  # Divide il dataset: 90% training, 10% valutazione. seed=42 garantisce la riproducibilità della divisione
print(f'Train: {len(dpo_split["train"])}  |  Eval: {len(dpo_split["test"])}')  # Stampa il numero di coppie in ciascun split

dpo_args = DPOConfig(
    output_dir='outputs/velvet-dpo-ner',       # Cartella dove salvare i checkpoint durante il training
    num_train_epochs=1,                        # Numero di epoche
    per_device_train_batch_size=4,             # Numero di coppie elaborate insieme per ogni passo di training
    gradient_accumulation_steps=2,             # Accumula i gradienti per 2 step prima di aggiornare i pesi
    learning_rate=1e-5,                        # Learning rate
    beta=0.5,                                  # Controlla quanto il modello si discosta dal modello di riferimento
    max_length=512,                            # Lunghezza massima in token di prompt+chosen/rejected (le sequenze più lunghe vengono troncate)
    fp16=False,                                # Disabilita float16
    bf16=True,                                 # Usa bfloat16
    logging_steps=10,                          # Stampa le metriche di training ogni 10 step
    save_steps=200,                            # Salva un checkpoint ogni 200 step
    eval_steps=200,                            # Esegue la valutazione sul set di valutazione ogni 200 step
    eval_strategy='steps',                     # Strategia di valutazione basata sul numero di step (e non sulle epoche)
    load_best_model_at_end=True,               # Al termine del training carica il checkpoint con la loss di valutazione più bassa
    report_to='none',                          # Disabilita il logging su servizi esterni
    remove_unused_columns=False,               # Mantiene tutte le colonne del dataset anche se non usate direttamente dal trainer
    warmup_steps=50,                           # Nei primi 50 step il learning rate cresce gradualmente da 0 fino al valore impostato
)

trainer = DPOTrainer(
    model=model_for_dpo,          # Modello da addestrare
    ref_model=None,               # Modello di riferimento — None perché con QLoRA non serve
    args=dpo_args,                # Configurazione del training definita sopra
    train_dataset=dpo_split['train'],  # Dataset di training
    eval_dataset=dpo_split['test'],    # Dataset di valutazione
    processing_class=tokenizer,        # Tokenizer
)

print('Avvio DPO training...')
trainer.train()  # Lancia il training

trainer.save_model('outputs/velvet-dpo-ner')        # Salva i pesi dell'adapter LoRA addestrato
tokenizer.save_pretrained('outputs/velvet-dpo-ner') # Salva il tokenizer nella stessa cartella
print('\nModello salvato in outputs/velvet-dpo-ner')

Train: 6194  |  Eval: 689


Adding EOS to train dataset:   0%|          | 0/6194 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6194 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/689 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/689 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 3}.


Avvio DPO training...


Step,Training Loss,Validation Loss
200,0.206153,0.103430
400,0.044084,0.081012
600,0.019852,0.073999


Modello salvato in outputs/velvet-dpo-ner


## **5. CONFRONTO BASELINE VS DPO**

**5.1 CARICAMENTO MODELLO BASE CON LoRA**

In questa fase ri-carichiamo Velvet base e gli carichiamo sopra l'adapter LoRA tramite il class method **PeftModel.from_pretrained** prende il modello base già caricato e ci applica sopra l'adapter LoRA salvato su disco (di cui passiamo il percorso).

Il metodo **merge_and_unload** fa due cose:

- merge: fonde i pesi LoRa (A·B) con i pesi originali W e fa ottenere un modello risultante che ha la stessa struttura di quello base ma con i pesi aggiornati dal DPO

- unload: Rimuove le strutture LoRA dalla memoria dopo aver aggiornato il modello


In [ ]:
del model_for_dpo, trainer  # Cancella il modello di training e il trainer dalla memoria Python
torch.cuda.empty_cache()    # Libera la memoria GPU

# CARICAMENTO DI VELVET BASE
base_model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

model_dpo = PeftModel.from_pretrained(base_model_eval, 'outputs/velvet-dpo-ner')  # Carica l'adapter LoRA addestrato e lo applica sopra il modello base
model_dpo = model_dpo.merge_and_unload()  # Fonde i pesi LoRA nei pesi originali e poi scarica la struttura LoRA dalla memoria
model_dpo = model_dpo.to('cuda')  # Sposta esplicitamente il modello sulla GPU  (perché merge_and_unload avrebbe potuto spostarlo sulla CPU)
model_dpo.eval()                  # Disabilita dropout e altri componenti usati solo durante il training, rendendo le predizioni deterministiche

print('\nModello DPO caricato.')

Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

Modello DPO caricato.
Device: cuda:0
Sat Mar 14 17:59:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   39C    P0            136W /  400W |    7018MiB /  81920MiB |      4%      Default |
|                                         |                        |             Disabled |
+----------

**5.2 INFERENZA DPO**

Rifacciamo l'inferenza utilizzando il prompt *P4_few_shot* sul dataset di testing (*all_samples_filtered* contiene ancora gli stessi esempi).

Calcoliamo le metriche con **compute_metrics_partial**, stampiamo a schermo la tabella e salviamo i risultati nel file *metrics_dpo.json*

In [ ]:
# Inferenza DPO completa
results_dpo = run_inference_prompt(
    all_samples_filtered, model_dpo, tokenizer,
    prompt_fn=PROMPTS['P4_few_shot'],
    batch_size=64
)

metrics_dpo = compute_metrics_partial(results_dpo)
print_metrics(metrics_dpo, 'DPO')

with open('outputs/metrics_dpo.json', 'w') as f:
    json.dump(metrics_dpo, f, indent=2)


Inference:   0%|          | 0/38 [00:00<?, ?it/s]

DPO


Tipo      Precision     Recall         F1
--------------------------------------------------
PER          0.5074     0.6256     0.5603
ORG          0.3455     0.2967     0.3192
LOC          0.5100     0.3423     0.4096
MICRO        0.4592     0.4177     0.4374


In [ ]:
# TABELLA COMPARATIVA FINALE
rows = []
for key in [*ENTITY_TYPES, 'MICRO']:  # Itera su PER, ORG, LOC e MICRO
    b = all_prompt_metrics['P4_few_shot'][key]  # Mette le metriche del modello baseline P4_few_shot nella variabile 'b'
    d = metrics_dpo[key]  # Mette le metriche del modello DPO nella variabile 'd'
    rows.append({
        'Tipo':   key,                                          # Tipo di entità o MICRO
        'Base P': b['precision'], 'Base R': b['recall'],  'Base F1': b['f1'],    # Metriche baseline
        'DPO P':  d['precision'], 'DPO R':  d['recall'],  'DPO F1':  d['f1'],   # Metriche DPO
        'ΔF1':    round(d['f1']-b['f1'], 4),                   # Delta F1
        'ΔP':     round(d['precision']-b['precision'], 4),     # Delta Precision
        'ΔR':     round(d['recall']-b['recall'], 4),           # Delta Recall
    })

df_cmp = pd.DataFrame(rows)  # Crea il DataFrame dalla lista di righe

def color_delta(v):  # Funzione di colorazione per i valori delta
    color = 'green' if v > 0 else ('red' if v < 0 else 'black')  # Verde per miglioramento, rosso per peggioramento, nero se invariato
    return f'color: {color}'  # Restituisce la stringa CSS per la colorazione

display(df_cmp.style.map(color_delta, subset=['ΔF1','ΔP','ΔR']))  # Mostra il DataFrame con i delta colorati (applica color_delta solo alle colonne dei delta)

df_cmp.to_csv('outputs/comparison_baseline_vs_dpo.csv', index=False)  # Salva il DataFrame in CSV
print('\nSalvato: outputs/comparison_baseline_vs_dpo.csv')

,Tipo,Base P,Base R,Base F1,DPO P,DPO R,DPO F1,ΔF1,ΔP,ΔR
0,PER,0.384200,0.418300,0.400500,0.507400,0.625600,0.560300,0.159800,0.123200,0.207300
1,ORG,0.138700,0.341200,0.197300,0.345500,0.296700,0.319200,0.121900,0.206800,-0.044500
2,LOC,0.270400,0.196700,0.227700,0.510000,0.342300,0.409600,0.181900,0.239600,0.145600
3,MICRO,0.224400,0.313300,0.261500,0.459200,0.417700,0.437400,0.175900,0.234800,0.104400


Salvato: outputs/comparison_baseline_vs_dpo.csv


In [ ]:
'''
# Analisi qualitativa errori residui
errors = [
    r for r in results_dpo
    if {(normalize(e['text']), e['type']) for e in r['gold_entities']} !=
       {(normalize(e['text']), e['type']) for e in r['pred_entities']}
]
print(f'Frasi ancora errate dopo DPO: {len(errors)} / {len(results_dpo)}')
for r in errors[:5]:
    print('\n---')
    print(f'  Frase: {r["sentence"][:100]}')
    print(f'  Gold:  {r["gold_entities"]}')
    print(f'  Pred:  {r["pred_entities"]}')
'''

Frasi ancora errate dopo DPO: 1677 / 2407

---
  Frase: Ricercatori britannici scoprono il primo antibiotico ricavato dagli insetti
  Gold:  []
  Pred:  [{'text': 'Ricercatori', 'type': 'PER'}, {'text': 'britannici', 'type': 'PER'}]

---
  Frase: Un gruppo di ricercatori dell' università di Nottingham nel Regno Unito , guidato da Simon Lee , ha 
  Gold:  [{'text': 'università di Nottingham', 'type': 'ORG'}, {'text': 'Regno Unito', 'type': 'LOC'}, {'text': 'Simon Lee', 'type': 'PER'}]
  Pred:  [{'text': 'Università di Nottingham', 'type': 'ORG'}]

---
  Frase: I risultati sono stati positivi e gli scienziati stanno analizzando la composizione chimica del cerv
  Gold:  []
  Pred:  [{'text': 'Istituto DESI', 'type': 'ORG'}, {'text': 'Istituto METI', 'type': 'ORG'}, {'text': 'Bangladesh', 'type': 'LOC'}]

---
  Frase: La scoperta potrebbe portare a nuove cure contro i batteri resistenti ai cocktail di farmaci .
  Gold:  []
  Pred:  [{'text': 'Istituto DESI', 'type': 'ORG'}, {'text': 'Istit

**CELLA PER BACKUP MANUALE SU DRIVE**.

In [ ]:
from datetime import datetime


# Configurazione Google Drive per salvare i file di output automaticamente
drive.mount('/content/drive', force_remount=False) # Se già montato non forziamo il re-mount
print('Drive montato.')


# Crea cartella con timestamp
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M')
DRIVE_BACKUP = f'/content/drive/MyDrive/tesi_velvet/backup_{timestamp}'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

# Lista di tutti i file da salvare
files_to_save = [
    # Risultati inferenza prompt comparison
    'outputs/results_P1_sintetico.json',
    'outputs/results_P2_medio.json',
    'outputs/results_P3_verboso.json',
    'outputs/results_P4_few_shot.json',
    'outputs/results_P5_lista.json',
    # Risultati inferenza training set
    'outputs/results_train_P4_few_shot.json',
    # Metriche
    'outputs/all_prompt_metrics.json',
    'outputs/prompt_comparison.csv',
    'outputs/metrics_dpo.json',
    'outputs/comparison_baseline_vs_dpo.csv',
    # Coppie DPO
    'outputs/dpo_pairs.json',
]

saved, skipped = [], []
for filepath in files_to_save:
    if os.path.exists(filepath):
        filename = os.path.basename(filepath)
        shutil.copy(filepath, f'{DRIVE_BACKUP}/{filename}')
        saved.append(filename)
    else:
        skipped.append(filepath)

print(f'Backup salvato in: {DRIVE_BACKUP}')
print(f'\nFile salvati ({len(saved)}):')
for f in saved:
    print(f'  ✓ {f}')
if skipped:
    print(f'\nFile non trovati ({len(skipped)}):')
    for f in skipped:
        print(f'  ✗ {f}')

# Salva anche il modello DPO
model_dir = 'outputs/velvet-dpo-ner'
if os.path.exists(model_dir):
    shutil.copytree(model_dir, f'{DRIVE_BACKUP}/velvet-dpo-ner', dirs_exist_ok=True)
    print(f'  velvet-dpo-ner (cartella modello)')
else:
    print(f'  velvet-dpo-ner (non trovata)')

Backup salvato in: /content/drive/MyDrive/tesi_velvet/backup_2026-03-14_20-21

File salvati (11):
  ✓ results_P1_sintetico.json
  ✓ results_P2_medio.json
  ✓ results_P3_verboso.json
  ✓ results_P4_few_shot.json
  ✓ results_P5_lista.json
  ✓ results_train_P4_few_shot.json
  ✓ all_prompt_metrics.json
  ✓ prompt_comparison.csv
  ✓ metrics_dpo.json
  ✓ comparison_baseline_vs_dpo.csv
  ✓ dpo_pairs.json
  ✓ velvet-dpo-ner (cartella modello)
